In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from pydantic import BaseModel
from typing import List, Optional, Dict, Any

In [ ]:
from model import llm

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# main.py
from graph import graph
import uuid
thread_id = str(uuid.uuid4())
def main():
    """
    Run chatbot with persistent memory using thread_id.
    Each thread_id maintains its own conversation history.
    """
    # Generate unique thread ID per conversation session
    # In production: use user_id, session_id, or conversation_id
    
    org_id = "test-org-2"
    print("Hotel Front Desk Bot (type 'quit' to exit)")
    print(f"Session: {thread_id[:8]}...")
    print("-" * 40)
    
    while True:
        message = input("\nYou: ").strip()
        if not message:
            continue
        if message.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break
        
        # Invoke graph with thread_id config for memory persistence
        result = graph.invoke(
            {"message": message, "org_id": org_id, "thread_id": thread_id},
            config={"configurable": {"thread_id": thread_id}}
        )
        
        # Extract response
        response = result.get("result", {}).get("response", "No response")
        print(f"\nBot: {response}")


if __name__ == "__main__":
    main()


Hotel Front Desk Bot (type 'quit' to exit)
Session: 24b1ebe1...
----------------------------------------
[load_context] thread_id: 24b1ebe1-1e86-4e79-a0b2-f81b9824f8c5
[load_context] loaded context: empty
[Router] LLM classification: CHAT (single greeting and introduction)
chat node executed
[save_to_redis] thread_id: 24b1ebe1-1e86-4e79-a0b2-f81b9824f8c5
[save_to_redis] message: hi i am biswa
[save_to_redis] response: Hello Biswa, welcome to our hotel. How can I assis
[save_to_redis] Saved user message
[save_to_redis] Saved assistant response

Bot: Hello Biswa, welcome to our hotel. How can I assist you today? Do you have a reservation with us or would you like to inquire about our rooms and services?
[load_context] thread_id: 24b1ebe1-1e86-4e79-a0b2-f81b9824f8c5
[load_context] loaded context: USER:hi i am biswa
ASSISTANT:Hello Biswa, welcome to our hotel. How can I assist you today? Do you h
[Router] LLM classification: LEAD_CAPTURE (User explicitly states 'i want to book', indicating

In [4]:
from utils import RedisMemoryService
memory = RedisMemoryService()
print(memory.get_context_string(thread_id))

USER:biswadipmandal99@gmail.com
ASSISTANT:Thank you for sharing your email address, Biswa. I've taken note of it as biswadipmandal99@gmail.com. I'll make sure to keep it confidential and only use it to send you relevant information regarding your booking.

So, to recap, I have:

* Your phone number: 6294357358
* Your email address: biswadipmandal99@gmail.com

Now, let's finalize the details of your booking. Could you please tell me:

* What dates are you planning to stay with us? (Check-in and check-out dates)
* How many guests will be accompanying you?
* What type of room are you looking for (single, double, suite, etc.)?
* Do you have any specific preferences, such as a city view or a particular floor?

We have various room options available, including:

* Deluxe Room: Starting from $100 per night
* Premium Room: Starting from $150 per night
* Suite: Starting from $250 per night

Please let me know your preferences, and I'll do my best to accommodate your needs and provide you with t

In [3]:
from utils import Vector_store_service
new = Vector_store_service("test-org-2")
new.retrieve_documents("payment methods", thresold=0.7)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 414.62it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'status': 'success',
 'query': 'payment methods',
 'total_retrieved': 5,
 'filtered_count': 5,
 'results': [{'id': 'e47bce54-4431-4720-a6be-2228d00beffb',
   'content': '. When payment is made in cash, this must be done in euros. The payment may not exceed the amount of 1,000 euros and, given that the rights must be exercised in accordance with the requirements of good faith, bills must be used that are appropriate to the amount to pay, meaning that the Hotel reserves the right to refuse high denomination bills if the amount to be paid is much lower. Payments with more than 50 coins of euros will also not be accepted. Users are obligated to pay the amount of contracted services when presented with the bill or in accordance with the agreed terms',
   'metadata': {'page': 10,
    'creator': 'Microsoft® Word 2016',
    'producer': 'iLovePDF',
    'page_label': '10',
    'total_pages': 36,
    'title': 'INTERNAL HOTEL PROCEDURE',
    'char_count': 2272,
    'creationdate': '2024-03-05T11:

In [1]:
from utils.redis_memory import RedisMemoryService
import uuid

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
# Cell 2: Initialize Redis Memory Service
redis_memory = RedisMemoryService(max_messages=20, ttl_seconds=3600)

# Test connection
try:
    redis_memory.client.ping()
    print("✅ Redis connection successful!")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")

✅ Redis connection successful!


In [26]:
# Cell 3: Create a test thread
test_thread_id = f"test-{uuid.uuid4().hex[:8]}"
print(f"Test thread ID: {test_thread_id}")

Test thread ID: test-97f2a8dc


In [27]:
# Cell 4: Add some test messages
redis_memory.add_message(test_thread_id, "user", "Hello, I want to book a room")
redis_memory.add_message(test_thread_id, "assistant", "Hello! I'd be happy to help you book a room. What dates are you looking for?")
redis_memory.add_message(test_thread_id, "user", "I need a room for March 15-17")
redis_memory.add_message(test_thread_id, "assistant", "Great! We have availability for March 15-17. Would you prefer a single or double room?")

print("✅ Messages added successfully!")

✅ Messages added successfully!


In [28]:
# Cell 5: Retrieve messages
messages = redis_memory.get_message(test_thread_id)
print(f"Retrieved {len(messages)} messages:\n")

for i, msg in enumerate(messages, 1):
    print(f"{i}. [{msg['role'].upper()}]: {msg['content']}")
    print(f"   Timestamp: {msg['timestamp']}\n")

Retrieved 4 messages:

1. [USER]: Hello, I want to book a room
   Timestamp: 2026-02-21T17:17:29.918020+00:00

2. [ASSISTANT]: Hello! I'd be happy to help you book a room. What dates are you looking for?
   Timestamp: 2026-02-21T17:17:29.920019+00:00

3. [USER]: I need a room for March 15-17
   Timestamp: 2026-02-21T17:17:29.920019+00:00

4. [ASSISTANT]: Great! We have availability for March 15-17. Would you prefer a single or double room?
   Timestamp: 2026-02-21T17:17:29.921024+00:00



In [29]:
# Cell 6: Get formatted context string (what LLM sees)
context = redis_memory.get_context_string(test_thread_id, limit=10)
print("📝 Context for LLM:\n")
print("-" * 50)
print(context)
print("-" * 50)

📝 Context for LLM:

--------------------------------------------------
USER:Hello, I want to book a room
ASSISTANT:Hello! I'd be happy to help you book a room. What dates are you looking for?
USER:I need a room for March 15-17
ASSISTANT:Great! We have availability for March 15-17. Would you prefer a single or double room?
--------------------------------------------------


In [8]:
# Cell 7: Test session state storage
test_state = {
    "intent": "LEAD_CAPTURE",
    "slots": {
        "check_in": "2024-03-15",
        "check_out": "2024-03-17",
        "room_type": None
    }
}

redis_memory.set_state(test_thread_id, test_state)
print("✅ State saved!")

# Retrieve state
retrieved_state = redis_memory.get_state(test_thread_id)
print(f"\n📦 Retrieved state:\n{retrieved_state}")

✅ State saved!

📦 Retrieved state:
{'intent': 'LEAD_CAPTURE', 'slots': {'check_in': '2024-03-15', 'check_out': '2024-03-17', 'room_type': None}}


In [10]:
# Cell 8: Test sliding window (max_messages limit)
overflow_thread = f"overflow-{uuid.uuid4().hex[:8]}"

# Add 25 messages (more than max_messages=20)
for i in range(25):
    redis_memory.add_message(overflow_thread, "user", f"Message {i+1}")

messages = redis_memory.get_message(overflow_thread)
print(f"Added 25 messages, retrieved {len(messages)} (max: 20)")
print(f"First message: {messages[0]['content']}")
print(f"Last message: {messages[-1]['content']}")

Added 25 messages, retrieved 20 (max: 20)
First message: Message 6
Last message: Message 25


In [11]:
# Cell 9: Cleanup test data
redis_memory.clear(test_thread_id)
redis_memory.clear(overflow_thread)

# Verify cleanup
messages_after = redis_memory.get_message(test_thread_id)
print(f"✅ Cleanup complete. Messages remaining: {len(messages_after)}")

✅ Cleanup complete. Messages remaining: 0


In [15]:
# Cell 10: Test with actual graph (end-to-end)
from graph import graph

thread_id = f"e2e-{uuid.uuid4().hex[:8]}"
org_id = "test-org-2"

# First message
result1 = graph.invoke({
    "message": "Hi, I want to know about your hotel",
    "org_id": org_id,
    "thread_id": thread_id
})
print(f"Response 1: {result1.get('result', {}).get('response', 'No response')}\n")

# Second message (should have context)
result2 = graph.invoke({
    "message": "What amenities do you offer?",
    "org_id": org_id,
    "thread_id": thread_id
})
print(f"Response 2: {result2.get('result', {}).get('response', 'No response')}\n")

# Check Redis context
print("📝 Stored context:")
print(redis_memory.get_context_string(thread_id))

[Router] LLM classification: INFORMATION_RETRIEVAL (User greeting followed by a request for general hotel information)
rag node executed
Response 1: I don't have that information. Let me connect you to a human agent.

[Router] LLM classification: INFORMATION_RETRIEVAL (User is asking for factual information about hotel amenities)
rag node executed
Response 2: The Hotel offers extra services such as tourist information, wake up service, storage of valuables in the Hotel's general safe, storage of baggage, and taxi calling service at no additional cost. Additionally, the Hotel can manage certain services beyond the establishment, such as car rental, excursions, restaurants, and other services related to the stay. Free services include Wi-Fi.

📝 Stored context:



In [21]:
# Cell 10: Test with actual graph (end-to-end)
# Restart kernel first or reload modules
import importlib
import graph as graph_module
importlib.reload(graph_module)
from graph import graph

from utils.redis_memory import RedisMemoryService
redis_memory = RedisMemoryService(max_messages=20, ttl_seconds=3600)

thread_id = f"e2e-{uuid.uuid4().hex[:8]}"
org_id = "test-org-2"

print(f"Thread ID: {thread_id}\n")

# First message
result1 = graph.invoke({
    "message": "Hi, I want to know about your hotel",
    "org_id": org_id,
    "thread_id": thread_id
})
print(f"Response 1: {result1.get('result', {}).get('response', 'No response')}\n")

# Check context after first message
print("📝 Context after message 1:")
print(redis_memory.get_context_string(thread_id))
print("-" * 50)

# Second message (should have context)
result2 = graph.invoke({
    "message": "What amenities do you offer?",
    "org_id": org_id,
    "thread_id": thread_id
})
print(f"\nResponse 2: {result2.get('result', {}).get('response', 'No response')}\n")

# Check Redis context
print("📝 Final stored context:")
print(redis_memory.get_context_string(thread_id))

Thread ID: e2e-fe1eb5b8

[load_context] thread_id: 
[Router] LLM classification: INFORMATION_RETRIEVAL (User is inquiring about general hotel information)
rag node executed
[save_to_redis] thread_id: 
[save_to_redis] No thread_id, skipping save
Response 1: I don't have that information. Let me connect you to a human agent.

📝 Context after message 1:

--------------------------------------------------
[load_context] thread_id: 
[Router] LLM classification: INFORMATION_RETRIEVAL (User is asking about hotel amenities, which is a factual information request)
rag node executed
[save_to_redis] thread_id: 
[save_to_redis] No thread_id, skipping save

Response 2: The Hotel offers extra services such as tourist information, wake up service, storage of valuables in the Hotel's general safe, storage of baggage, and taxi calling service at no additional cost. Additionally, the Hotel can manage certain services beyond the establishment, such as car rental, excursions, restaurants, and other servic

In [25]:
print(redis_memory.get_context_string(test_thread_id))

In [18]:
# Debug cell: Test save_to_redis directly
from utils.redis_memory import RedisMemoryService
from graph import redis_memory as graph_redis

# Check if it's the same instance
local_redis = RedisMemoryService(max_messages=20, ttl_seconds=3600)

test_id = "debug-test-123"

# Test direct save
graph_redis.add_message(test_id, "user", "Test message from graph redis")
local_redis.add_message(test_id, "user", "Test message from local redis")

print("From graph_redis:", graph_redis.get_context_string(test_id))
print("From local_redis:", local_redis.get_context_string(test_id))

# Cleanup
graph_redis.clear(test_id)

From graph_redis: USER:Test message from graph redis
USER:Test message from local redis
From local_redis: USER:Test message from graph redis
USER:Test message from local redis


In [31]:
# Test save_to_redis function directly with dummy state
from graph import save_to_redis, redis_memory
import uuid

# Create a dummy state matching GraphState structure
test_thread_id = f"dummy-{uuid.uuid4().hex[:8]}"
print(f"Test thread ID: {test_thread_id}")

dummy_state = {
    "message": "Hello, I want to book a room",
    "org_id": "test-org-2",
    "thread_id": test_thread_id,
    "intent": "LEAD_CAPTURE",
    "context": "",
    "result": {
        "response": "Sure! I can help you book a room. What dates are you looking for?"
    }
}

print("\n📤 Input state:")
print(f"  message: {dummy_state['message']}")
print(f"  response: {dummy_state['result']['response']}")
print(f"  thread_id: {dummy_state['thread_id']}")

# Call save_to_redis
print("\n🔄 Calling save_to_redis...")
result_state = save_to_redis(dummy_state)

# Check what's in Redis
print("\n📥 Checking Redis:")
context = redis_memory.get_context_string(test_thread_id)
print(f"Context:\n{context}")

messages = redis_memory.get_message(test_thread_id)
print(f"\nTotal messages stored: {len(messages)}")

for i, msg in enumerate(messages, 1):
    print(f"  {i}. [{msg['role']}]: {msg['content'][:50]}...")

# Cleanup
redis_memory.clear(test_thread_id)
print("\n✅ Test complete, cleaned up.")

Test thread ID: dummy-e94c3344

📤 Input state:
  message: Hello, I want to book a room
  response: Sure! I can help you book a room. What dates are you looking for?
  thread_id: dummy-e94c3344

🔄 Calling save_to_redis...
[save_to_redis] thread_id: dummy-e94c3344
[save_to_redis] message: Hello, I want to book a room
[save_to_redis] response: Sure! I can help you book a room. What dates are y
[save_to_redis] Saved user message
[save_to_redis] Saved assistant response

📥 Checking Redis:
Context:
USER:Hello, I want to book a room
ASSISTANT:Sure! I can help you book a room. What dates are you looking for?

Total messages stored: 2
  1. [user]: Hello, I want to book a room...
  2. [assistant]: Sure! I can help you book a room. What dates are y...

✅ Test complete, cleaned up.


In [1]:
# Test full graph with debug output
import importlib
import graph as graph_module
importlib.reload(graph_module)
from graph import graph, redis_memory

import uuid
thread_id = f"full-{uuid.uuid4().hex[:8]}"
org_id = "test-org-2"

print(f"Thread ID: {thread_id}")
print("=" * 60)

# Check graph structure
print("\n📊 Graph nodes:", list(graph.nodes.keys()))
print("=" * 60)

# Invoke graph
result = graph.invoke({
    "message": "What rooms do you have?",
    "org_id": org_id,
    "thread_id": thread_id
})

print("=" * 60)
print(f"\n✅ Response: {result.get('result', {}).get('response', 'No response')[:100]}...")

# Check Redis
print("\n📥 Checking Redis after graph execution:")
context = redis_memory.get_context_string(thread_id)
print(f"Context: {context if context else '(empty)'}")

messages = redis_memory.get_message(thread_id)
print(f"Total messages: {len(messages)}")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Thread ID: full-37a99dd6

📊 Graph nodes: ['__start__', 'load_context', 'intent_router', 'rag_node', 'lead_node', 'issue_node', 'handoff_node', 'chat_node', 'save_memory']
[load_context] thread_id: full-37a99dd6
[load_context] loaded context: empty
[Router] LLM classification: INFORMATION_RETRIEVAL (User is inquiring about room types, which is a factual information request)
rag node executed


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 583.60it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[save_to_redis] thread_id: full-37a99dd6
[save_to_redis] message: What rooms do you have?
[save_to_redis] response: I don't have that information. Let me connect you 
[save_to_redis] Saved user message
[save_to_redis] Saved assistant response

✅ Response: I don't have that information. Let me connect you to a human agent....

📥 Checking Redis after graph execution:
Context: USER:What rooms do you have?
ASSISTANT:I don't have that information. Let me connect you to a human agent.
Total messages: 2


In [34]:
# Check GraphState schema
from schema import GraphState
print("GraphState fields:")
print(GraphState.__annotations__)

GraphState fields:
{'message': <class 'str'>, 'messages': typing.List[typing.Dict[str, str]], 'org_id': <class 'str'>, 'intent': typing.Optional[str], 'confidence': typing.Optional[float], 'next_node': typing.Optional[str], 'context': typing.Optional[str], 'result': typing.Optional[typing.Dict[str, typing.Any]]}
